In [ ]:
#!pip install boto3

### boto3
We'll be using boto3 library. This is the AWS SDK for Python. This SDK allows us to write software that makes use of AWS services. It can look for credentials in multiple places.

Although not recommended, credentials can be directly specified also:

```
bedrock_client = boto3.client(aws_access_key_id="",
                       aws_secret_access_key="",
                       region_name="us-east-1", service_name='bedrock')
```

 For this example, we have stored it as credentials file in .aws folder:

```
[default]
aws_access_key_id = <key_here>
aws_secret_access_key = <secret_here>
```


In [ ]:
import boto3
bedrock_client = boto3.client(region_name="us-east-1",
                       service_name='bedrock')
bedrock_client.list_foundation_models()

In [ ]:
bedrock_client.get_foundation_model(modelIdentifier='amazon.titan-tg1-large')

### To invoke models, we need to go through bedrock-runtime service (and not the client one)

In [ ]:
bedrock = boto3.client(region_name="us-east-1",service_name='bedrock-runtime')

### We'll play around with Titan Express model from Amazon

**About model**

Amazon Titan Text Express has a context length of up to 8,000 tokens, making it well-suited for a wide range of advanced, general language tasks such as open-ended text generation and conversational chat, as well as support within Retrieval Augmented Generation (RAG).

```
{
    "inputText": string,
    "textGenerationConfig": {
        "temperature": float,  
        "topP": float,
        "maxTokenCount": int,
        "stopSequences": [string]
    }
}
```

In [ ]:
import json

Let us start by asking factual question without any parameters specified:

In [ ]:
body = json.dumps({"inputText": "What is the capital of America?"})

In [ ]:
response = bedrock.invoke_model(body=body, modelId="amazon.titan-text-express-v1")

In [ ]:
print(response)

As we can see from above, response is streaming body. Let us read it now:

In [ ]:
response_body = json.loads(response.get('body').read())

In [ ]:
response_body

Print just the output text:

In [ ]:
print(response_body["results"][0]["outputText"])

### Now let us get to question that can have different answers

We'll ask about Big Bang Theory. Model can tell us about actual theory or about the show with same name. We'll first try without any parameters and hence can expect random answer

In [ ]:
prompt = "Tell me about big bang theory"

In [ ]:
body = json.dumps({"inputText": prompt})
response = bedrock.invoke_model(body=body, modelId="amazon.titan-text-express-v1")
response_body = json.loads(response.get('body').read())
print(response_body["results"][0]["outputText"])

**It told us about scientific theory. Let us try again. We can get different response about theory itself or about the show**

In [ ]:
body = json.dumps({"inputText": prompt})
response = bedrock.invoke_model(body=body, modelId="amazon.titan-text-express-v1")
response_body = json.loads(response.get('body').read())
print(response_body["results"][0]["outputText"])

**Response was different. It was about the show**

Let us now start **specifying parameters** to make our intentions clear. We want to get **repeatable answers**, so we'll set the value of temperature to zero and topP to low value (as it cannot be zero):

In [ ]:
body = json.dumps({
    "inputText": prompt,
    "textGenerationConfig": {
        "temperature": 0,
        "topP": 0.01,  
        "maxTokenCount": 1024
    }
})
response = bedrock.invoke_model(body=body, modelId="amazon.titan-text-express-v1")
response_body = json.loads(response.get('body').read())
print(response_body["results"][0]["outputText"])

**Analysis:**

* As we specified maxTokenCount, we got bit longer response
* Response was about the show


In [ ]:
body = json.dumps({
    "inputText": prompt,
    "textGenerationConfig": {
        "temperature": 0,
        "topP": 0.01,  
        "maxTokenCount": 1024
    }
})
response = bedrock.invoke_model(body=body, modelId="amazon.titan-text-express-v1")
response_body = json.loads(response.get('body').read())
print(response_body["results"][0]["outputText"])

**Analysis:**

* As we specified same maxTokenCount, we got bit longer response
* We specified low values for temperature and topP, we got similar answer